# 05 · TSP case study: subtour elimination by separation
**Branch-and-Cut Workshop · UANL & Tecnológico de Monterrey** (Prof. Angélica Salazar, Prof. Vincent Boyer)

*Activity 6 · about 18 min.*

> **How to use this notebook.** Run the cells from top to bottom (Shift+Enter).
> Cells marked **✍️ TODO** are for you. Cells marked **🤖 AI prompt** contain a prompt you can paste into
> Gemini (the ✨ button in Colab), ChatGPT, Claude… **Always read and test the code the AI gives you.**

In [ ]:
# These notebooks use the PuLP 3 API (PuLP 4 changed it). Colab has no PuLP: install PuLP 3 (it ships with CBC).
import importlib.metadata, subprocess, sys
try:
    _pulp_version = importlib.metadata.version("pulp")
except importlib.metadata.PackageNotFoundError:
    _pulp_version = None
if _pulp_version is None or int(_pulp_version.split(".")[0]) >= 4:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pulp>=2.7,<4"], check=True)
import pulp
if int(pulp.__version__.split(".")[0]) >= 4:
    raise RuntimeError("PuLP 3 was just installed, but PuLP 4 is still loaded: restart the kernel "
                       "(Colab: Runtime > Restart session) and run this cell again.")

import itertools, math, time
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

def make_instance(n, seed=0):
    rng = np.random.default_rng(seed)
    P = rng.uniform(0, 100, size=(n, 2))
    E = list(itertools.combinations(range(n), 2))
    d = {e: float(np.hypot(*(P[e[0]] - P[e[1]]))) for e in E}
    return P, E, d

def plot_edges(P, xvals, title="", ax=None):
    ax = ax or plt.gca()
    for (i, j), v in xvals.items():
        if v > 1e-6:
            ax.plot(*zip(P[i], P[j]), color="#0072CE" if v > 0.99 else "#C8102E",
                    lw=2 if v > 0.99 else 1.2, ls="-" if v > 0.99 else "--")
    ax.scatter(P[:, 0], P[:, 1], color="k", s=20, zorder=3)
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])

## Notation
| Symbol | Meaning | In the code |
|---|---|---|
| $G = (V, E)$ | complete undirected graph, $V = \{0, \dots, n-1\}$ cities, $\lvert E \rvert = n(n-1)/2$ | `range(n)`, `E` |
| $e = \{i, j\}$ | edge (unordered pair of cities) | tuple `(i, j)` with `i < j` |
| $c_e$ | length of edge $e$ (Euclidean distance) | `d[e]` |
| $x_e \in \{0,1\}$ | 1 if the tour uses edge $e$ | `x[e]` |
| $x(F) = \sum_{e \in F} x_e$ | total value on an edge set $F$ | `lpSum(x[e] for e in F)` |
| $\delta(i)$ | edges incident to city $i$ | `[e for e in E if i in e]` |
| $\delta(S)$ | edges with **exactly one** end in $S$ (the *cut* of $S$) | `(e[0] in S) != (e[1] in S)` |
| $E(S)$ | edges with **both** ends in $S$ | `e[0] in S and e[1] in S` |

## Model (DFJ)
$$
\begin{aligned}
\min\; & \sum_{e \in E} c_e x_e \\
\text{s.t. } & x(\delta(i)) = 2 && \forall i \in V && \text{(degree: enter and leave each city once)}\\
& x(\delta(S)) \ge 2 && \forall S \subset V,\ 2 \le |S| \le n-2 && \text{(SEC, cut form)}\\
& x_e \in \{0, 1\} && \forall e \in E
\end{aligned}
$$
Equivalent **packing form** of the SEC: $x(E(S)) \le |S| - 1$. Summing the degree equations over $i \in S$ gives
$2|S| = 2\,x(E(S)) + x(\delta(S))$, so the two forms are the same constraint.

*Example:* a subtour on $S = \{0, 4, 9\}$ uses 3 edges inside $S$ and none leaving it:
$x(E(S)) = 3 > 2 = |S| - 1$ and $x(\delta(S)) = 0 < 2$, so the SEC for this $S$ is violated.

We start with the **degree constraints only** and add SECs when needed.

In [ ]:
def build_model(n, E, d, integer=True):
    m = pulp.LpProblem("TSP", pulp.LpMinimize)
    cat = "Binary" if integer else "Continuous"
    x = {e: pulp.LpVariable(f"x_{e[0]}_{e[1]}", 0, 1, cat=cat) for e in E}
    m += pulp.lpSum(d[e] * x[e] for e in E)
    for i in range(n):
        m += pulp.lpSum(x[e] for e in E if i in e) == 2
    return m, x

def add_sec(m, x, S, E):
    S = set(S)
    m += pulp.lpSum(x[e] for e in E if (e[0] in S) != (e[1] in S)) >= 2

## ✍️ Activity 6a — be the separation oracle (16 cities)
You play the role of the separation algorithm. **You** decide which constraint is missing:

1. **Generate** an instance (number of cities, seed) and solve it with the degree constraints only.
2. Every city has exactly two edges, yet the solution may not be a tour. Find the group(s) of cities $S$ that
   are **cut off** from the others (*subtours*) and type them in the **cut cell**: for each $S$ the cut
   $x(\delta(S)) \ge 2$ ("at least two edges must leave $S$") is added, the model is re-solved and re-drawn.
3. **Re-run the same cut cell** with your next set(s), until you get one tour.
4. **Share** on Miro (frame *6 – TSP results*): your final picture (right-click → copy image) and the output of `summary()`.

*Strategy question:* do you cut **one** subtour per round, or **all** of them? Who needs the fewest rounds?

*Keep this in mind for 6b:* **how** did you find a subtour in the picture? Write down your procedure in words.

In [ ]:
# @title Helper functions for Activity 6a (run this cell; no need to read it)
COLOR_COMPONENTS = False        # True colours each connected component differently
COLORS = ["#0072CE", "#C8102E", "#007A4C", "#E07B00", "#7A3E9D", "#555555", "#00A3AD", "#B5838D"]

def new_instance(n=16, seed=5):
    """Random Euclidean TSP with n cities (same seed = same cities), solved with the degree constraints only."""
    global N_CITIES, SEED
    N_CITIES, SEED = n, seed
    reset()
    solve_and_show()

def reset():
    """Start again from the degree-constraint model (no SECs)."""
    global P, E, d, model, x, log, cuts_added, components
    P, E, d = make_instance(N_CITIES, SEED)
    model, x = build_model(N_CITIES, E, d)
    log, cuts_added, components = [], [], []

def solve_and_show():
    """Solve the current model and draw the solution with node numbers."""
    global components
    model.solve(pulp.PULP_CBC_CMD(msg=False))
    chosen = [e for e in E if x[e].value() > 0.5]
    components = [sorted(c) for c in nx.connected_components(nx.Graph(chosen))]
    cost = pulp.value(model.objective)
    log.append({"round": len(log) + 1, "cost": cost, "subtours": len(components)})
    if COLOR_COMPONENTS:
        color = {v: COLORS[k % len(COLORS)] for k, S in enumerate(components) for v in S}
    else:
        color = {v: COLORS[0] for v in range(N_CITIES)}
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    for (i, j) in chosen:
        ax.plot(*zip(P[i], P[j]), color=color[i], lw=2.5)
    for v, (px, py) in enumerate(P):
        ax.scatter(px, py, s=300, color="white", edgecolor=color[v], lw=2, zorder=3)
        ax.text(px, py, str(v), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_title(f"round {len(log)}: cost {cost:.1f}, {len(cuts_added)} SEC(s) added so far")
    ax.set_aspect("equal"); ax.axis("off"); plt.show()
    if len(components) == 1:
        print("One single tour: this is the optimal TSP tour!")
    elif COLOR_COMPONENTS:
        for k, S in enumerate(components):
            print(f"component {k}: {S}")

def add_cut(S):
    """Add the subtour elimination constraint x(delta(S)) >= 2 if it is violated by the current solution."""
    S = set(S)
    if not S or len(S) >= N_CITIES or not S <= set(range(N_CITIES)):
        print("S must be a non-empty proper subset of the nodes 0 ..", N_CITIES - 1); return False
    crossing = sum(x[e].value() for e in E if (e[0] in S) != (e[1] in S))
    if crossing >= 2 - 1e-6:
        print(f"{crossing:.0f} edges already leave S: this SEC is NOT violated. Pick another S."); return False
    add_sec(model, x, S, E)
    cuts_added.append(sorted(S))
    print(f"Added SEC #{len(cuts_added)}: at least 2 edges must leave {sorted(S)} (currently {crossing:.0f}).")
    return True

def cut_and_resolve(sets):
    """Add the SEC of every violated set in `sets`, then re-solve and re-draw."""
    added = [add_cut(S) for S in sets]
    if any(added):
        solve_and_show()
    else:
        print("No new cut added: the solution did not change.")

def summary():
    print(f"{N_CITIES} cities, seed {SEED}: {len(log)} solves, {len(cuts_added)} SECs added")
    for r in log:
        print(f"  round {r['round']}: cost {r['cost']:7.1f}, {r['subtours']} component(s)")
    print("sets S cut:", cuts_added)

In [ ]:
# Generate the cities (change the seed for another instance) and solve with the degree constraints only
new_instance(n=16, seed=5)

In [ ]:
# ✍️ THE CUT CELL: re-run this same cell once per round.
# Type the subtour(s) you see in the last picture: one set, or several at once.
SETS = [[]]               # e.g. [[0, 7, 12]]   or   [[0, 7, 12], [4, 6, 13]]
cut_and_resolve(SETS)

In [ ]:
summary()                 # paste this output on Miro with your final picture

**Discuss.** Why does the cost never decrease from one round to the next? Why is every round's cost a valid
**lower bound** on the optimal tour?

*Start again?* Re-run the `new_instance` cell to go back to round 1 and try the other strategy.

## ✍️ Activity 6b — from your eyes to an oracle
**Step 0 (Miro, 2 min).** How did you detect a subtour? Answer in graph language:

* The chosen edges form a graph $G^* = (V, \{e : x^*_e = 1\})$. What does a subtour correspond to in $G^*$?
* Which classical algorithm finds it? How much does it cost for $n$ cities?
* When is the solution a tour? (How many … does $G^*$ have?)

**Step 1.** Turn your answer into code: complete `find_subtours(n, chosen)` **without** calling networkx.
*Hint:* start from an unvisited city and follow edges until you cannot reach anything new; repeat.

In [ ]:
def find_subtours(n, chosen):
    """chosen: list of edges (i, j) with x_e = 1.
    Return the connected components of the graph ({0..n-1}, chosen) as a list of sorted lists."""
    # ✍️ TODO: your oracle (graph search: DFS or BFS)
    raise NotImplementedError

In [ ]:
# test: two triangles -> [[0, 1, 2], [3, 4, 5]]; a 6-cycle -> one component
print(find_subtours(6, [(0, 1), (1, 2), (0, 2), (3, 4), (4, 5), (3, 5)]))
print(find_subtours(6, [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (0, 5)]))

**Step 2.** Plug your oracle into the loop: `tsp_integer_loop` does what you did by hand, cutting **every**
subtour your function returns in each round. Check that it finds the same optimal cost as in 6a.

In [ ]:
def tsp_integer_loop(n, seed=0, verbose=True):
    P, E, d = make_instance(n, seed)
    m, x = build_model(n, E, d)
    t0, rounds, n_secs = time.time(), 0, 0
    while True:
        rounds += 1
        m.solve(pulp.PULP_CBC_CMD(msg=False))
        sol = {e: x[e].value() for e in E}
        comps = find_subtours(n, [e for e in E if sol[e] > 0.5])      # your oracle
        if verbose:
            print(f"round {rounds}: cost {pulp.value(m.objective):8.2f}, {len(comps)} component(s)")
        if len(comps) == 1:
            break
        for S in comps:
            add_sec(m, x, S, E); n_secs += 1
    return dict(n=n, rounds=rounds, secs=n_secs, cost=pulp.value(m.objective),
                time=time.time() - t0, P=P, sol=sol)

res = tsp_integer_loop(N_CITIES, seed=SEED)      # same instance as 6a: same optimal cost?
plot_edges(res["P"], res["sol"], f"optimal tour, cost {res['cost']:.1f}"); plt.show()

**Step 3 — scaling.** Run the loop for $n = 20, 40, 60$ and fill the Miro table (rounds, SECs, cost, time).

In [ ]:
table = []
for n in [20, 40, 60]:
    r = tsp_integer_loop(n, seed=1, verbose=False)
    table.append(r)
    print(f"n={n:3d}  rounds={r['rounds']:3d}  SECs={r['secs']:4d}  cost={r['cost']:8.2f}  time={r['time']:6.1f}s")

### 🤖 Step 4 (if time) — when your oracle is not enough: fractional solutions
The integer loop re-solves a full MIP each round. Branch-and-Cut instead strengthens the **LP** at every node,
where $x^*$ is fractional. Then the support graph $\{e : x^*_e > 0\}$ can be **connected** while some $S$ still has
$x^*(\delta(S)) < 2$: your graph search finds nothing, yet a SEC is violated.
The right oracle is a **minimum cut** of the support graph with capacities $x^*_e$.

**🤖 AI prompt:**
```text
Using networkx, write separate_sec_fractional(n, x_star) where x_star maps edges (i, j), i<j, of a
complete graph on n nodes to LP values. Build the support graph with capacity x_star[e] for x_star[e] > 1e-6,
and if it is disconnected return each component as a violated set S. Otherwise compute a global min
cut with nx.stoer_wagner (weight='capacity'); if its value < 2 - 1e-6 return [S] for the smaller side,
else return []. Explain why x(delta(S)) < 2 means a violated subtour elimination constraint.
```

In [ ]:
def separate_sec_fractional(n, x_star):
    # TODO: paste / write your min-cut separation here; return a list of node sets S
    raise NotImplementedError

### Step 5 (if time) — how strong is the LP?
Compare the **degree-only LP**, the **subtour LP** (degree + all SECs, found by your separation), and the optimum.

In [ ]:
def subtour_lp(n, seed=0, max_rounds=200):
    P, E, d = make_instance(n, seed)
    m, x = build_model(n, E, d, integer=False)
    m.solve(pulp.PULP_CBC_CMD(msg=False))
    z_deg = pulp.value(m.objective)
    for r in range(max_rounds):
        sol = {e: x[e].value() for e in E}
        sets = separate_sec_fractional(n, sol)
        if not sets:
            break
        for S in sets:
            add_sec(m, x, S, E)
        m.solve(pulp.PULP_CBC_CMD(msg=False))
    return z_deg, pulp.value(m.objective), r, P, {e: x[e].value() for e in E}

for n in [20, 40, 60]:
    z_deg, z_sec, rounds, P, sol = subtour_lp(n, seed=1)
    z_opt = next(r["cost"] for r in table if r["n"] == n)
    print(f"n={n:3d}  degree LP {z_deg:8.2f} ({100*(z_opt-z_deg)/z_opt:5.2f}% gap)   "
          f"subtour LP {z_sec:8.2f} ({100*(z_opt-z_sec)/z_opt:5.2f}% gap, {rounds} rounds)   OPT {z_opt:8.2f}")

plot_edges(P, sol, "subtour LP solution (dashed = fractional)"); plt.show()

**Discuss.** Is the subtour LP solution a tour? What would branch-and-cut do next?
(Answer: branch on a fractional edge, and keep separating SECs at every node.)

## Optional — real Branch-and-Cut with Gurobi lazy constraints
Requires `pip install gurobipy` (the free pip license handles up to 2,000 variables, i.e. $n \le 63$ cities;
use your academic license for more).

In [ ]:
try:
    import gurobipy as gp
    from gurobipy import GRB

    def tsp_gurobi(n, seed=1):
        P, E, d = make_instance(n, seed)
        env = gp.Env(empty=True); env.setParam("OutputFlag", 0); env.start()
        m = gp.Model(env=env); m.Params.LazyConstraints = 1
        x = m.addVars(E, obj=d, vtype=GRB.BINARY)
        m.addConstrs(gp.quicksum(x[e] for e in E if i in e) == 2 for i in range(n))
        m._x, m._n, m._lazy = x, n, 0

        def cb(model, where):
            if where == GRB.Callback.MIPSOL:
                vals = model.cbGetSolution(model._x)
                G = nx.Graph([e for e in model._x if vals[e] > 0.5])
                for S in nx.connected_components(G):
                    if len(S) < model._n:
                        model.cbLazy(gp.quicksum(model._x[i, j] for i, j in model._x
                                                 if (i in S) != (j in S)) >= 2)
                        model._lazy += 1

        t0 = time.time(); m.optimize(cb)
        print(f"n={n}: cost {m.ObjVal:.2f}, {m._lazy} lazy SECs, {m.NodeCount:.0f} B&B nodes, {time.time()-t0:.1f}s")

    for n in [20, 40, 60]:
        tsp_gurobi(n)
except Exception as err:
    print("Gurobi not available here:", err)